In [1]:
# ===== FULL REBUILD (new session) + STAGE 16 FOLD 1: MOBILENETV2 ONLY =====
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'   # MUST precede tensorflow import
import random, gc
import numpy as np
import pandas as pd
import tensorflow as tf

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print(f"Seed {SEED} set, TF {tf.__version__}, tf.keras module: {tf.keras.__name__}")
assert 'tf_keras' in tf.keras.__name__, "STOP: Keras 3 active, not legacy. Restart before continuing."

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mob_pre
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, CSVLogger
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, recall_score, confusion_matrix
print("Imports ready")

FINAL_CLASSES = ['bcc', 'bkl', 'df', 'melanoma', 'nevus', 'vasc']
NUM_CLASSES = 6
IMG_SIZE, BATCH_SIZE = 224, 32
N_FOLDS, CURRENT_FOLD = 5, 1
AUG = dict(rotation_range=20, width_shift_range=0.1, height_shift_range=0.1,
           horizontal_flip=True, zoom_range=0.1)
print("Config loaded")

CV_ASSIGN_PATH = '/kaggle/input/datasets/ab0y04/cvfoldassignments/cv_fold_assignments.csv'
assert os.path.exists(CV_ASSIGN_PATH), f"STOP: file not found at {CV_ASSIGN_PATH}, check the dataset is attached"
cv_assignments = pd.read_csv(CV_ASSIGN_PATH)
print(f"Loaded cv_fold_assignments.csv: {len(cv_assignments):,} rows (expect 23,836)")
assert len(cv_assignments) == 23836, "STOP: row count mismatch"

def build_fold_split(cv_assignments, fold_num, seed=42):
    test_df = cv_assignments[cv_assignments['fold'] == fold_num].reset_index(drop=True)
    remaining = cv_assignments[cv_assignments['fold'] != fold_num].reset_index(drop=True)
    remaining = remaining.copy()
    fallback = pd.Series('unlinked_' + remaining.index.astype(str), index=remaining.index)
    remaining['_split_key'] = remaining['group_id'].fillna(fallback)
    groups = remaining.groupby('_split_key')['label'].first().reset_index()
    tr_groups, va_groups = train_test_split(groups, test_size=0.15, stratify=groups['label'], random_state=seed)
    train_df = remaining[remaining['_split_key'].isin(tr_groups['_split_key'])].drop(columns=['_split_key']).reset_index(drop=True)
    val_df = remaining[remaining['_split_key'].isin(va_groups['_split_key'])].drop(columns=['_split_key']).reset_index(drop=True)
    return train_df, val_df, test_df

train_df, val_df, test_df = build_fold_split(cv_assignments, CURRENT_FOLD, seed=SEED)
print(f"\n===== FOLD {CURRENT_FOLD} SPLIT =====")
print(f"Train {len(train_df):,} | Val {len(val_df):,} | Test {len(test_df):,}")

test_groups = set(test_df['group_id'].dropna())
train_groups = set(train_df['group_id'].dropna())
val_groups = set(val_df['group_id'].dropna())
assert test_groups.isdisjoint(train_groups) and test_groups.isdisjoint(val_groups) and train_groups.isdisjoint(val_groups), \
    "STOP: FOLD 1 LEAKAGE detected"
print("Leakage check: PASS")

cls = np.array(FINAL_CLASSES)
cw = compute_class_weight('balanced', classes=cls, y=train_df['label'])
w_map = {c: w for c, w in zip(FINAL_CLASSES, cw)}
train_df['sample_weight'] = train_df['label'].map(w_map)
print(f"Fold {CURRENT_FOLD} class_weight:", {c: round(w,3) for c,w in zip(cls, cw)})
print("(expect to match the run that trained custom+eff: bcc 1.184, df 16.284, nevus 0.309)")

def make_fold_gens(preprocess_fn):
    train_idg = ImageDataGenerator(preprocessing_function=preprocess_fn, **AUG)
    eval_idg  = ImageDataGenerator(preprocessing_function=preprocess_fn)
    common = dict(x_col='image_path', y_col='label', target_size=(IMG_SIZE,IMG_SIZE),
                  batch_size=BATCH_SIZE, class_mode='categorical', classes=FINAL_CLASSES)
    tr = train_idg.flow_from_dataframe(train_df, shuffle=True, seed=SEED, weight_col='sample_weight', **common)
    va = eval_idg.flow_from_dataframe(val_df, shuffle=False, **common)
    te = eval_idg.flow_from_dataframe(test_df, shuffle=False, **common)
    return tr, va, te

def build_pretrained(base_class, num_classes=6, shape=(224,224,3)):
    base = base_class(include_top=False, weights='imagenet', input_shape=shape)
    model = Sequential([base, GlobalAveragePooling2D(), Dense(256,activation='relu'),
                          Dropout(0.3), Dense(num_classes,activation='softmax')])
    return model, base

def macro_specificity(y_true, y_pred, n_classes):
    cm = confusion_matrix(y_true, y_pred, labels=range(n_classes))
    total = cm.sum(); specs = []
    for i in range(n_classes):
        tp = cm[i,i]; fn = cm[i,:].sum()-tp; fp = cm[:,i].sum()-tp
        tn = total-tp-fn-fp
        specs.append(tn/(tn+fp) if (tn+fp)>0 else np.nan)
    return np.nanmean(specs)

print(f"\n===== Fold {CURRENT_FOLD} setup verified. Training MobileNetV2 only. =====\n")

fold_results = []

# ---------- MOBILENETV2 ----------
try:
    print(f"{'='*60}\nFOLD {CURRENT_FOLD}: MobileNetV2\n{'='*60}")
    tr, va, te = make_fold_gens(mob_pre)
    print("class_indices:", te.class_indices)

    model, base = build_pretrained(MobileNetV2)
    log_file = f'/kaggle/working/cv_f{CURRENT_FOLD}_mob_log.csv'

    base.trainable = False
    model.compile(Adam(1e-3), 'categorical_crossentropy', ['accuracy'])
    model.fit(tr, validation_data=va, epochs=10, callbacks=[CSVLogger(log_file, append=False)], verbose=1)
    print("Phase 1 sanity:", model.evaluate(te, verbose=0))

    base.trainable = True
    model.compile(Adam(1e-5), 'categorical_crossentropy', ['accuracy'])
    cbs = [EarlyStopping(monitor='val_accuracy', patience=7, restore_best_weights=True),
           ModelCheckpoint(f'/kaggle/working/cv_f{CURRENT_FOLD}_mob.keras', monitor='val_accuracy', save_best_only=True),
           CSVLogger(log_file, append=True)]
    model.fit(tr, validation_data=va, epochs=60, callbacks=cbs, verbose=1)

    y_true = np.asarray(te.classes)
    y_prob = model.predict(te, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)
    np.savez(f'/kaggle/working/cv_f{CURRENT_FOLD}_preds_mob.npz', y_true=y_true, y_pred=y_pred, y_prob=y_prob)

    reloaded = load_model(f'/kaggle/working/cv_f{CURRENT_FOLD}_mob.keras')
    verify_acc = reloaded.evaluate(te, verbose=0)[1]
    live_acc = accuracy_score(y_true, y_pred)
    print(f"Checkpoint verify: reloaded acc {verify_acc:.4f} vs live acc {live_acc:.4f}  match={abs(verify_acc-live_acc)<1e-3}")
    del reloaded

    fold_results.append(dict(fold=CURRENT_FOLD, arch='mob', accuracy=live_acc,
        macro_f1=f1_score(y_true,y_pred,average='macro'),
        macro_auc=roc_auc_score(np.eye(NUM_CLASSES)[y_true], y_prob, average='macro', multi_class='ovr'),
        macro_sensitivity=recall_score(y_true,y_pred,average='macro'),
        macro_specificity=macro_specificity(y_true,y_pred,NUM_CLASSES), n_test=len(y_true)))
    print(f"RESULT: {fold_results[-1]}")
    del model, base; gc.collect(); tf.keras.backend.clear_session()
except Exception as e:
    print(f"!!! FOLD {CURRENT_FOLD} MobileNetV2 FAILED: {type(e).__name__}: {e}")
    import traceback; traceback.print_exc()
    gc.collect(); tf.keras.backend.clear_session()

# ---------- SAVE, checking whether custom+eff's results file already exists somewhere ----------
existing = os.popen("find /kaggle/input -iname 'cv_f1_results.csv' 2>/dev/null").read().strip()
results_path = '/kaggle/working/cv_f1_results.csv'
new_df = pd.DataFrame(fold_results)

if existing:
    print(f"\nFound a prior cv_f1_results.csv at: {existing}")
    prior_df = pd.read_csv(existing)
    combined = pd.concat([prior_df, new_df], ignore_index=True).drop_duplicates(subset=['fold','arch'], keep='last')
    combined.to_csv(results_path, index=False)
    print(f"Merged with prior results, {len(combined)} total rows.")
else:
    print(f"\nNo prior cv_f1_results.csv found under /kaggle/input.")
    print("This means custom+eff's results from the earlier session were NOT uploaded to a dataset.")
    print("Saving MobileNetV2's result alone. You will need to manually re-add custom+eff's numbers")
    print("when assembling the final fold 1 summary, they are recorded in this conversation:")
    print("  custom: acc=0.4773 macro_f1=0.3371 macro_auc=0.7653")
    print("  eff:    acc=0.7200 macro_f1=0.6015 macro_auc=0.9139")
    new_df.to_csv(results_path, index=False)

print(f"\n{'='*60}\nFOLD {CURRENT_FOLD} STATUS\n{'='*60}")
for r in fold_results:
    print(f"{r['arch']:8} acc={r['accuracy']:.4f}  macro_f1={r['macro_f1']:.4f}  macro_auc={r['macro_auc']:.4f}")
print(f"\nSaved: {results_path}")
print("Still needed for fold 1: res")

2026-08-12 19:25:48.276095: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1786562748.438331      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1786562748.485755      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1786562748.875558      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1786562748.875597      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1786562748.875600      23 computation_placer.cc:177] computation placer alr

Seed 42 set, TF 2.19.0, tf.keras module: tf_keras.api._v2.keras
Imports ready
Config loaded
Loaded cv_fold_assignments.csv: 23,836 rows (expect 23,836)

===== FOLD 1 SPLIT =====
Train 16,121 | Val 2,882 | Test 4,833
Leakage check: PASS
Fold 1 class_weight: {np.str_('bcc'): np.float64(1.184), np.str_('bkl'): np.float64(1.514), np.str_('df'): np.float64(16.284), np.str_('melanoma'): np.float64(0.88), np.str_('nevus'): np.float64(0.309), np.str_('vasc'): np.float64(15.442)}
(expect to match the run that trained custom+eff: bcc 1.184, df 16.284, nevus 0.309)

===== Fold 1 setup verified. Training MobileNetV2 only. =====

FOLD 1: MobileNetV2
Found 16121 validated image filenames belonging to 6 classes.
Found 2882 validated image filenames belonging to 6 classes.
Found 4833 validated image filenames belonging to 6 classes.
class_indices: {'bcc': 0, 'bkl': 1, 'df': 2, 'melanoma': 3, 'nevus': 4, 'vasc': 5}


I0000 00:00:1786562835.987307      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1786562835.994331      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


9406464/9406464 [==============================] - 2s 0us/step
Epoch 1/10


I0000 00:00:1786562844.608832      68 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1786562846.595108      66 service.cc:152] XLA service 0x7c32ec389520 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1786562846.595147      66 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1786562846.595151      66 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1786562846.819980      66 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


504/504 [==============================] - 522s 1s/step - loss: 1.4457 - accuracy: 0.4634 - val_loss: 1.2014 - val_accuracy: 0.5739
Epoch 2/10
504/504 [==============================] - 334s 663ms/step - loss: 1.1587 - accuracy: 0.5286 - val_loss: 1.1015 - val_accuracy: 0.5978
Epoch 3/10
504/504 [==============================] - 327s 649ms/step - loss: 1.0661 - accuracy: 0.5526 - val_loss: 1.0651 - val_accuracy: 0.6041
Epoch 4/10
504/504 [==============================] - 327s 649ms/step - loss: 0.9831 - accuracy: 0.5743 - val_loss: 1.1934 - val_accuracy: 0.5510
Epoch 5/10
504/504 [==============================] - 329s 653ms/step - loss: 0.9853 - accuracy: 0.5711 - val_loss: 1.2285 - val_accuracy: 0.5271
Epoch 6/10
504/504 [==============================] - 330s 654ms/step - loss: 0.9277 - accuracy: 0.5871 - val_loss: 1.0229 - val_accuracy: 0.5992
Epoch 7/10
504/504 [==============================] - 328s 651ms/step - loss: 0.8776 - accuracy: 0.5993 - val_loss: 0.9554 - val_accuracy:

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/datasets/ab0y04/k-foold-validation-fold-1/cv_f1_results.csv\n/kaggle/input/notebooks/ab0y04/skin-lesion-imagenet/cv_f1_results.csv'